In [186]:
import os
import enum

from dataclasses import dataclass
from typing import List, Dict, Any, Optional, Tuple, Callable

import torch
from torch import Tensor
from torch import quasirandom as t_qrand

import botorch
from botorch import fit as b_fit
from botorch import models as b_models
from botorch import acquisition as b_acq
from botorch import optim as b_optim
from botorch import test_functions as b_tfuncs
from botorch.utils import transforms as b_transf
from botorch.posteriors import gpytorch as bp_gpytorch

import gpytorch
from gpytorch import likelihoods as g_lh
from gpytorch import constraints as g_constraints
from gpytorch import kernels as g_kern
from gpytorch import mlls as g_mlls

device: torch.device = torch.device("mps" if torch.mps.is_available() else "cpu")
tensor_dtype: torch.dtype = torch.float32
runs: int = os.environ.get("SMOKE_TEST", 30)

dim: int = 3
num_samples: int = 25

lb: float = -100
ub: float = 100

confidence_level: float = 2

batch_size: int = 256
max_iterations: int = 60
max_cholesky_size: int = float("inf")

In [187]:
@dataclass
class BOState():
    max_iterations: int
    current_iteration: int = 1

    num_restarts: int = 10
    raw_samples: int = 512

    eta_1: float = 0.2
    eta_2: float = 0.8

    gamma_inc: float = 1.2
    gamma_red: float = 0.8

    delta_max: float = 1.5
    delta_min: float = 0.1

    def is_exceeded(self):
        return self.current_iteration > self.max_iterations
    def increment(self):
        print(f"Current iteration: {self.current_iteration}")
        self.current_iteration = self.current_iteration + 1

In [188]:
ackley: b_tfuncs.Ackley = b_tfuncs.Ackley(
    dim=dim,
    negate=False,
    ).to(
        device=device, 
        dtype=tensor_dtype
        )
ackley.bounds[0, :].fill_(lb)
ackley.bounds[1, :].fill_(ub)

def eval_ackley(x: Tensor):
    return ackley(x)

In [189]:
def create_surrogate(
        X: Tensor,
        Y: Tensor,
        noise_interval: g_constraints.Interval = g_constraints.Interval(1e-6, 1e-4),
        matern_smoothness: float = 2.5,
) -> Tuple[b_models.SingleTaskGP, g_lh.GaussianLikelihood]:
    likelihood: g_lh.GaussianLikelihood = g_lh.GaussianLikelihood(
        noise_constraint=noise_interval,
    )
    kernel: g_kern.ScaleKernel = g_kern.ScaleKernel(
        base_kernel=g_kern.MaternKernel(
            nu=matern_smoothness,
        )
    )
    model: b_models.SingleTaskGP = b_models.SingleTaskGP(
        train_X=X,
        train_Y=Y,
        likelihood=likelihood,
        covar_module=kernel,
    )
    mll: g_mlls.ExactMarginalLogLikelihood = g_mlls.ExactMarginalLogLikelihood(
        likelihood=likelihood,
        model=model
    )
    return (model, mll)

def sample_candidate_positions(
    dim: int,
    batch_size: int,
    scramble: Optional[bool] = True, 
) -> Tensor:
    sobol: t_qrand.SobolEngine = t_qrand.SobolEngine(
        dimension=dim,
        scramble=scramble,
        
    )
    return sobol.draw(n=batch_size).to(device=device, dtype=tensor_dtype).requires_grad_(True) * 2 - 1

def get_ucb(
        model: b_models.SingleTaskGP,
        X: Tensor,
        beta: float = confidence_level,
    ) -> Tensor:
    posterior: bp_gpytorch.GPyTorchPosterior = model.posterior(X=X)
    return posterior.mean + beta * torch.sqrt(posterior.variance)

def get_lcb(
        model: b_models.SingleTaskGP, 
        X: Tensor,
        beta: float = confidence_level,
        ) -> Tensor:
    posterior: bp_gpytorch.GPyTorchPosterior = model.posterior(X=X)
    return posterior.mean - beta * torch.sqrt(posterior.variance)


def get_center_next(
        model: b_models.SingleTaskGP,
        state: BOState,
        X: Tensor,
        D: Tensor,
        delta_t: float,
    ) -> Tensor:
    lcb_XD: Tensor = get_lcb(
        model=model,
        X=X+D,
    )
    eucl_mask: Tensor = (torch.linalg.norm(D, dim=1) <= delta_t)
    abs_XD_proposed: Tensor = torch.abs(X+D)

    # Achkley bounds
    constraint_mask: Tensor = (abs_XD_proposed <= ub).all(dim=1)
    valid_mask: Tensor = constraint_mask & eucl_mask
    if not valid_mask.any():
        return torch.zeros_like(D)

    masked: Tensor = torch.where(valid_mask.unsqueeze(1), lcb_XD, float("inf"))

    return D[torch.argmin(input=masked, dim=0).squeeze()]

def get_accuracy_ratio(
        model: b_models.SingleTaskGP,

        X_prev: Tensor,
        D_next: Tensor,

        eval_func: Callable,
    ) -> float:

    new: Tensor = X_prev + D_next

    posterior_XD: bp_gpytorch.GPyTorchPosterior = model.posterior(X=new.unsqueeze(0))
    posterior_X: bp_gpytorch.GPyTorchPosterior  = model.posterior(X=X_prev.unsqueeze(0))
    
    a_num: Tensor = (eval_func(new) - eval_func(X_prev))
    a_denom: Tensor = posterior_XD.mean - posterior_X.mean
    
    ratio: Tensor = a_num/a_denom
    if ratio.shape[1] != 1:
        return 0
    return ratio.item()

def get_next_candidates(
        state: BOState,
        accuracy_ratio: float,    
        delta_t: float,
    ) -> Tensor:
    
    if accuracy_ratio < state.eta_1:
        return max(delta_t * state.gamma_red, state.delta_min)
    if state.eta_1 < accuracy_ratio < state.eta_2:
        return delta_t
    if state.eta_2 < accuracy_ratio:
        return min(delta_t * state.gamma_inc, state.delta_max)
    

In [190]:
x_safe: Tensor = torch.rand(
    size=[num_samples, dim],
    requires_grad=True
).to(device=device, dtype=tensor_dtype)
y_safe: Tensor = torch.tensor(
    [eval_ackley(x_safe[i, :]) for i in range(num_samples)],
).to(device=device, dtype=tensor_dtype).unsqueeze(-1)

state: BOState = BOState(
    max_iterations=max_iterations,
)

delta_t: float = (state.delta_max + state.delta_min) / 2.0

while not state.is_exceeded():
    state.increment()
    model, mll = create_surrogate(
        X=x_safe.detach(),
        Y=y_safe.detach(),
    )
    
    with gpytorch.settings.max_cholesky_size(max_cholesky_size):
        b_fit.fit_gpytorch_mll(mll=mll)

        x_prev = x_safe[-1, :]

        d_candidates: Tensor = sample_candidate_positions(
            dim=x_safe.shape[1],
            batch_size=batch_size, 
            scramble=True
        )
        d_new: Tensor = get_center_next(
            model=model,
            state=state,
            X=x_prev.unsqueeze(0), 
            D=d_candidates,
            delta_t=delta_t,
        )
        accuracy_ratio: float = get_accuracy_ratio(
            model=model,
            X_prev=x_prev,
            D_next=d_new,
            eval_func=eval_ackley
        )
        delta_t_new: float = get_next_candidates(
            state=state,
            accuracy_ratio=accuracy_ratio,
            delta_t=delta_t,
        )
        next_candidates: Tensor = (x_prev + d_new)
        if next_candidates.dim() == 1:
            next_candidates: Tensor = next_candidates.unsqueeze(0)
        y_next: Tensor = torch.tensor(
            [eval_ackley(x) for x in next_candidates], 
            dtype=tensor_dtype, 
            device=device
        ).unsqueeze(-1)

        x_safe: Tensor = torch.cat(
            (x_safe, next_candidates),
            dim=0,
        )
        y_safe: Tensor = torch.cat(
            (y_safe, y_next), 
            dim=0,
        )

        delta_t: float = delta_t_new

        print(f"minimum y value: {torch.amin(y_safe)}")



Current iteration: 1


/var/folders/95/n78ktpt12l71xwnt2mjjtd0w0000gn/T/ipykernel_78938/669232798.py:15: InputDataWarning: The model inputs are of type torch.float32. It is strongly recommended to use double precision in BoTorch, as this improves both precision and stability and can help avoid numerical errors. See https://github.com/meta-pytorch/botorch/discussions/1444
  model: b_models.SingleTaskGP = b_models.SingleTaskGP(


minimum y value: 2.4040708541870117
Current iteration: 2


/var/folders/95/n78ktpt12l71xwnt2mjjtd0w0000gn/T/ipykernel_78938/669232798.py:15: InputDataWarning: The model inputs are of type torch.float32. It is strongly recommended to use double precision in BoTorch, as this improves both precision and stability and can help avoid numerical errors. See https://github.com/meta-pytorch/botorch/discussions/1444
  model: b_models.SingleTaskGP = b_models.SingleTaskGP(


minimum y value: 2.4040708541870117
Current iteration: 3


...

minimum y value: 0.26435375213623047
Current iteration: 59


/var/folders/95/n78ktpt12l71xwnt2mjjtd0w0000gn/T/ipykernel_78938/669232798.py:15: InputDataWarning: The model inputs are of type torch.float32. It is strongly recommended to use double precision in BoTorch, as this improves both precision and stability and can help avoid numerical errors. See https://github.com/meta-pytorch/botorch/discussions/1444
  model: b_models.SingleTaskGP = b_models.SingleTaskGP(


minimum y value: 0.26435375213623047
Current iteration: 60


/var/folders/95/n78ktpt12l71xwnt2mjjtd0w0000gn/T/ipykernel_78938/669232798.py:15: InputDataWarning: The model inputs are of type torch.float32. It is strongly recommended to use double precision in BoTorch, as this improves both precision and stability and can help avoid numerical errors. See https://github.com/meta-pytorch/botorch/discussions/1444
  model: b_models.SingleTaskGP = b_models.SingleTaskGP(


minimum y value: 0.26435375213623047


In [191]:
import plotly.graph_objects as go
import numpy as np
import torch

def plot_safeopt_objective_plotly(y_safe: torch.Tensor, num_initial_samples: int):
    # 1. Extract only the objective values evaluated DURING the loop
    # We flatten the array so Plotly can read it easily as a 1D list
    y_evaluated = y_safe[num_initial_samples:].detach().cpu().numpy().flatten()
    iterations = np.arange(1, len(y_evaluated) + 1)

    # 2. Initialize the Plotly figure
    fig = go.Figure()

    # 3. Add the SafeOpt objective line (expect high fluctuations!)
    fig.add_trace(
        go.Scatter(
            x=iterations, 
            y=y_evaluated, 
            mode='lines', 
            name='SafeOpt',
            line=dict(color='#1f77b4')
        )
    )

    # 4. Add the dashed red line for the 'minimum value' benchmark (0.0 for Ackley)
    fig.add_trace(
        go.Scatter(
            x=[iterations, iterations[-1]], 
            y=[0.0, 0.0], 
            mode='lines', 
            name='minimum value',
            line=dict(color='firebrick', dash='dash')
        )
    )

    # 5. Format the layout to match the paper's style
    fig.update_layout(
        title='SafeOpt Performance (Ackley Function)',
        xaxis_title='Iteration',
        yaxis_title='Objective',
        template='plotly_white', # Gives a clean background similar to matplotlib
        legend=dict(
            yanchor="top",
            y=0.99,
            xanchor="right",
            x=0.99
        )
    )
    
    fig.show(renderer="vscode") 

plot_safeopt_objective_plotly(
    y_safe=y_safe,
    num_initial_samples=num_samples
)